# Regressão Linear Simples — Exemplo 01

Prever o **valor total da nota fiscal** com base na **quantidade total de itens vendidos**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Conexão com o Banco de Dados PostgreSQL

In [ ]:
usuario = "datadt_data_analytics"
senha = "DataAnalytics$100"
host = "postgresql-datadt.alwaysdata.net"
porta = "5432"
banco = "datadt_digital_corporativo"

engine = create_engine(
    f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{banco}"
)

## 2. Consulta SQL

Cada linha representa uma nota fiscal. `X` = quantidade total de itens, `Y` = valor total da nota.

In [ ]:
sql = """
SELECT 
    nf.id AS id_nota_fiscal,
    SUM(inf.quantidade) AS quantidade_total_itens,
    SUM(inf.quantidade * inf.valor_unitario) AS valor_total_nota
FROM vendas.nota_fiscal nf
JOIN vendas.item_nota_fiscal inf 
    ON inf.id_nota_fiscal = nf.id
GROUP BY nf.id
ORDER BY nf.id;
"""

## 3. Carregando os Dados

In [ ]:
df = pd.read_sql(sql, engine)

print("Primeiras linhas da base:")
print(df.head())

print("\nInformações da base:")
print(df.info())

print("\nResumo estatístico:")
print(df.describe())

## 4. Tratamento Básico dos Dados

In [ ]:
df = df.dropna()

df = df[
    (df["quantidade_total_itens"] > 0) &
    (df["valor_total_nota"] > 0)
]

print("Quantidade de registros após tratamento:")
print(len(df))

## 5. Definindo X e Y

`X` precisa estar em formato de matriz (dois colchetes). `Y` é a variável alvo.

In [ ]:
X = df[["quantidade_total_itens"]]
y = df["valor_total_nota"]

## 6. Divisão em Treino e Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 7. Treinando o Modelo

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)

## 8. Coeficientes do Modelo

In [ ]:
intercepto = modelo.intercept_
coeficiente = modelo.coef_[0]

print("Modelo treinado:")
print(f"Intercepto: {intercepto:.2f}")
print(f"Coeficiente: {coeficiente:.2f}")

print("\nEquação da regressão:")
print(f"valor_total_nota = {intercepto:.2f} + {coeficiente:.2f} * quantidade_total_itens")

## 9. Fazendo Previsões

In [ ]:
y_pred = modelo.predict(X_test)

resultado = pd.DataFrame({
    "quantidade_total_itens": X_test["quantidade_total_itens"],
    "valor_real": y_test,
    "valor_previsto": y_pred
})

print("Comparação entre valor real e valor previsto:")
print(resultado.head())

## 10. Avaliação do Modelo

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Métricas de avaliação:")
print(f"MAE  - Erro médio absoluto: {mae:.2f}")
print(f"MSE  - Erro quadrático médio: {mse:.2f}")
print(f"R²   - Coeficiente de determinação: {r2:.4f}")

## 11. Visualização dos Dados e da Reta de Regressão

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(X, y, label="Dados reais")
plt.plot(X, modelo.predict(X), label="Reta de regressão")

plt.title("Regressão Linear Simples")
plt.xlabel("Quantidade total de itens")
plt.ylabel("Valor total da nota fiscal")
plt.legend()
plt.grid(True)

arquivo_grafico = "grafico_regressao_exemplo01.png"
plt.savefig(arquivo_grafico, dpi=300, bbox_inches="tight")
plt.show()

print(f"\nGráfico salvo em: {arquivo_grafico}")

## 12. Simulação de Previsão

Prever o valor de uma nota com 10 itens vendidos.

In [ ]:
nova_quantidade = pd.DataFrame({
    "quantidade_total_itens": [10]
})

valor_estimado = modelo.predict(nova_quantidade)

print("Simulação:")
print(f"Para uma nota com 10 itens, o valor estimado é R$ {valor_estimado[0]:.2f}")